# Master Analysis Orchestrator

**Objective:** This notebook serves as the central controller for the entire AMR-SSI analysis pipeline.

It executes all other analysis notebooks in a predefined, sequential order. This ensures reproducibility and a fully automated workflow from data preprocessing to final output generation.

**Execution Flow:**
1.  **Setup:** Initializes logging and sets up system paths.
2.  **Execution:** Runs each notebook in the `notebook_pipeline` list using Papermill.
3.  **Completion:** Confirms the end of the pipeline run.

In [25]:
# === 1. SETUP: IMPORTS & PATH CONFIGURATION ===
import os
import sys
import papermill as pm
from datetime import datetime

# Add the project root to the Python path to allow importing from 'utils'
# This assumes the notebook is in the 'notebooks' directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project Root: {project_root}")
print("System path configured.")

Project Root: c:\Users\Marion Korir\code\amr-ssi-python-analysis
System path configured.


In [26]:
# === 2. INITIALIZE LOGGING ===
from utils.logging_utils import setup_logging

# Initialize the main pipeline logger
# All subsequent logs in the pipeline will use this instance
logger = setup_logging(file_prefix="master_orchestrator")

logger.info("Master orchestrator notebook started.")

INFO: Logging initialized. Log file: logs\master_orchestrator_20250921_134352.log
INFO: Master orchestrator notebook started.
INFO: Master orchestrator notebook started.


### Notebook Execution Order

The following notebooks will be executed in sequence. Each notebook represents a distinct phase of the analysis, ensuring a logical and traceable workflow.

1.  `01_Data_Preprocessing.ipynb`: Loads raw data, performs cleaning, and handles deduplication.
2.  `02_Analysis_SSI_Incidence.ipynb`: Conducts the meta-analysis of Surgical Site Infection (SSI) incidence.
3.  `03_Analysis_AMR_Proportions.ipynb`: Runs meta-analyses on pathogen-specific antimicrobial resistance proportions.
4.  `04_Analysis_Temporal_Trends.ipynb`: Performs meta-regression to analyze temporal trends in AMR.
5.  `05_Analysis_Outcomes.ipynb`: Analyzes secondary outcomes like mortality and risk factors.
6.  `06_Analysis_Narrative_Synthesis.ipynb`: Generates descriptive summaries for qualitative data, length of stay, and costs.

In [27]:
# === 3. DEFINE THE PIPELINE ===

# List of notebooks to be executed in order.
# The paths are relative to the 'notebooks' directory.
notebook_pipeline = [
    '01_Data_Preprocessing.ipynb',
    '02_Analysis_SSI_Incidence.ipynb',
    '03_Analysis_AMR_Proportions.ipynb',
    '04_Analysis_Temporal_Trends.ipynb',
    '05_Analysis_Outcomes.ipynb',
    '06_Analysis_Narrative_Synthesis.ipynb'
]

# Base directory for input and output notebooks
base_dir = os.getcwd() 
output_dir = os.path.join(base_dir, "executed")
os.makedirs(output_dir, exist_ok=True)

logger.info(f"Defined notebook execution pipeline with {len(notebook_pipeline)} notebooks.")
print(f"Notebooks will be run from: {base_dir}")
print(f"Executed notebooks will be saved to: {output_dir}")

INFO: Defined notebook execution pipeline with 6 notebooks.
Notebooks will be run from: c:\Users\Marion Korir\code\amr-ssi-python-analysis\notebooks
Executed notebooks will be saved to: c:\Users\Marion Korir\code\amr-ssi-python-analysis\notebooks\executed
Notebooks will be run from: c:\Users\Marion Korir\code\amr-ssi-python-analysis\notebooks
Executed notebooks will be saved to: c:\Users\Marion Korir\code\amr-ssi-python-analysis\notebooks\executed


In [28]:
# === 4. EXECUTE THE PIPELINE ===
# This cell iterates through the defined pipeline and runs each notebook.
# The output of each run (an executed notebook) is saved in the 'executed' sub-directory.

logger.info("Starting pipeline execution.")
execution_start_time = datetime.now()

for notebook in notebook_pipeline:
    input_path = os.path.join(base_dir, notebook)
    output_path = os.path.join(output_dir, f"executed_{notebook}")
    
    if not os.path.exists(input_path):
        logger.warning(f"Notebook not found: {input_path}. Skipping.")
        continue

    try:
        logger.info(f"--- Executing {notebook} ---")
        notebook_start_time = datetime.now()        
        pm.execute_notebook(
            input_path=input_path,
            output_path=output_path,
            # You can pass parameters here if needed, for example:
            # parameters=dict(master_log_file=logger.handlers[0].baseFilename)
        )
        notebook_duration = datetime.now() - notebook_start_time
        logger.info(f"--- Finished {notebook} in {notebook_duration}. Output saved to {output_path} ---")

    except Exception as e:
        logger.error(f"An error occurred while executing {notebook}: {e}", exc_info=True)
        # Decide if you want to stop the pipeline on error or continue
        break 

total_duration = datetime.now() - execution_start_time
logger.info(f"Pipeline execution finished in {total_duration}.")

INFO: Starting pipeline execution.
INFO: --- Executing 01_Data_Preprocessing.ipynb ---
INFO: --- Executing 01_Data_Preprocessing.ipynb ---


Executing:   0%|          | 0/7 [00:00<?, ?cell/s]

INFO: --- Finished 01_Data_Preprocessing.ipynb in 0:00:04.626990. Output saved to c:\Users\Marion Korir\code\amr-ssi-python-analysis\notebooks\executed\executed_01_Data_Preprocessing.ipynb ---
INFO: Pipeline execution finished in 0:00:04.639586.
INFO: Pipeline execution finished in 0:00:04.639586.


---
## Pipeline Execution Complete
---